# 02 — Polling-place data linkage audit

**Research notebook — no geocoded establishments are bundled.** This notebook verifies the relationship between election sections and physical polling establishments across election years/rounds. Electoral section and zone numbers are *not geographic coordinates*.

Before drawing spatial conclusions, inspect link quality, changing establishment allocations, coordinate precision and coverage. Required fields are listed in the [data dictionary](../docs/data-dictionary.md).

In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

section_path = repo_root / "data/processed/section_turnout.csv"
mapping_path = repo_root / "data/processed/section_place_mapping.csv"
places_path = repo_root / "data/processed/polling_places.csv"

expected_paths = [section_path, mapping_path, places_path]
for path in expected_paths:
    print(f"{path.name}: {'available' if path.exists() else 'not yet provided'}")

keys = ["year", "round", "municipality_code", "electoral_zone", "section_id"]


## 1. Validate record keys and linkage quality

Each section should appear once per election/round in normalized turnout data, and should map to at most one verified establishment for that same election/round. A missing mapping must be reported explicitly instead of imputing geography from a section ID.

In [ ]:
audit = None
if not all(p.exists() for p in expected_paths):
    print("Linkage audit skipped until all three normalized CSV files exist.")
else:
    codes = {"municipality_code": "string", "electoral_zone": "string", "section_id": "string"}
    sections = pd.read_csv(section_path, dtype=codes)
    mapping = pd.read_csv(mapping_path, dtype={**codes, "establishment_id": "string"})
    places = pd.read_csv(
        places_path, dtype={"municipality_code": "string", "establishment_id": "string"}
    )
    required_sections = set(keys + ["eligible", "turnout", "absent"])
    required_mapping = set(keys + ["establishment_id"])
    required_places = {"municipality_code", "establishment_id", "latitude", "longitude"}
    for name, data, required in [
        ("section_turnout", sections, required_sections),
        ("section_place_mapping", mapping, required_mapping),
        ("polling_places", places, required_places),
    ]:
        missing = required - set(data.columns)
        if missing:
            raise ValueError(f"{name} missing columns: {sorted(missing)}")

    if sections[keys].isna().any().any() or sections.duplicated(keys).any():
        raise ValueError("Invalid or duplicate section turnout keys")
    if mapping[keys].isna().any().any() or mapping.duplicated(keys).any():
        raise ValueError("Invalid or duplicate section-to-place mapping keys")
    if places[["municipality_code", "establishment_id"]].isna().any().any():
        raise ValueError("Null establishment identifiers")
    if places.duplicated(["municipality_code", "establishment_id"]).any():
        raise ValueError("Duplicate establishment identifiers within municipality")
    if ((sections["eligible"] <= 0) |
        (sections["turnout"] < 0) |
        (sections["absent"] < 0) |
        (sections["eligible"] != sections["turnout"] + sections["absent"])).any():
        raise ValueError("Inconsistent section-level counts")

    linked = sections.merge(mapping, on=keys, how="left", validate="one_to_one", indicator=True)
    print(f"Sections: {len(sections):,}")
    print(f"Unmapped sections: {(linked['_merge'] == 'left_only').sum():,}")
    audit = linked.merge(
        places,
        on=["municipality_code", "establishment_id"],
        how="left",
        validate="many_to_one",
    )
    mapped_without_coords = audit["establishment_id"].notna() & (
        audit["latitude"].isna() | audit["longitude"].isna()
    )
    print(f"Mapped sections without coordinates: {mapped_without_coords.sum():,}")


## 2. Aggregate section counts to verified establishments

The aggregate requires a verified section-to-establishment mapping and valid coordinates. This notebook does **not** estimate voter travel distances or individual behavior.

In [ ]:
if audit is None:
    print("Polling-place aggregation skipped: source files missing.")
else:
    lon = pd.to_numeric(audit["longitude"], errors="coerce")
    lat = pd.to_numeric(audit["latitude"], errors="coerce")
    valid_coordinates = lat.between(-90, 90) & lon.between(-180, 180)
    valid = audit.loc[
        audit["establishment_id"].notna() & valid_coordinates
    ].copy()
    if valid.empty:
        print("No mapped sections with valid coordinates.")
    else:
        aggregated = valid.groupby(
            ["year", "round", "municipality_code", "establishment_id"], as_index=False
        ).agg(eligible=("eligible", "sum"),
              absent=("absent", "sum"),
              turnout=("turnout", "sum"),
              latitude=("latitude", "first"),
              longitude=("longitude", "first"))
        aggregated["abstention_rate"] = aggregated["absent"] / aggregated["eligible"]
        print(f"Verified polling-place/round rows: {len(aggregated):,}")
        print("Review geocoding precision and linkage completeness before analysis.")


## Next steps

- Review section-to-establishment link evidence and mapping validity dates.
- Build geodesic or locally projected kNN spatial weights at the establishment level.
- Compare raw establishment proximity against independent access/travel-time data before interpreting physical accessibility.
- Fit a multilevel binomial specification with municipality and polling-establishment effects after data quality gates are met.